# HalluGuard model showcase

HalluGuard checks whether a supplied passage supports a claim. Enter the two texts to get a verdict and the model's support score. A passage that contradicts the claim or leaves out a required detail can lead to an unsupported verdict.

### Run on Kaggle

1. Open or import this notebook. If you're viewing the published Kaggle copy, choose **Copy & Edit**.
2. Under **Input → Add Input → Models**, attach the published HalluGuard model variation and version you want to test. A copied notebook may already have that input attached.
3. Run the setup cell once, then run the sample table or edit the custom passage and claim below.

### Run locally from GitHub

Download the model files from the HalluGuard model page on Kaggle. Place them together in a directory named `hallucination_deberta_v3_v4` beside this notebook, or enter their directory in `MODEL_DIR`. Start Jupyter in the notebook's folder. If needed, install the dependencies with `python -m pip install notebook torch transformers pandas sentencepiece`.

Visitors need access to the Kaggle model version to run it. The notebook does not require the training and test datasets.

In [ ]:
import json
from pathlib import Path

import pandas as pd
import torch
from IPython.display import display
from transformers import AutoModelForSequenceClassification, AutoTokenizer

# Leave as None to find a single attached export, or set the exact folder path.
MODEL_DIR = None

INFERENCE_FILES = (
    "config.json",
    "model.safetensors",
    "tokenizer.json",
    "tokenizer_config.json",
    "threshold_config.json",
)
BASE_MODEL = "MoritzLaurer/DeBERTa-v3-large-mnli-fever-anli-ling-wanli"


def find_model_dir(input_root):
    matches = []
    for config_path in sorted(input_root.rglob("threshold_config.json")):
        folder = config_path.parent
        if not all((folder / name).is_file() for name in INFERENCE_FILES):
            continue
        config = json.loads(config_path.read_text(encoding="utf-8"))
        if config.get("model_name") == BASE_MODEL:
            matches.append(folder)

    if not matches:
        raise FileNotFoundError(
            "No complete HalluGuard export found. Attach it in Kaggle or set MODEL_DIR."
        )
    if len(matches) > 1:
        options = "\n".join(str(folder) for folder in matches)
        raise ValueError(f"Set MODEL_DIR to one of these model versions:\n{options}")
    return matches[0]


local_export = Path("hallucination_deberta_v3_v4")
if MODEL_DIR is not None:
    model_dir = Path(MODEL_DIR)
elif local_export.is_dir():
    model_dir = local_export
else:
    model_dir = find_model_dir(Path("/kaggle/input"))

missing = [name for name in INFERENCE_FILES if not (model_dir / name).is_file()]
if missing:
    raise FileNotFoundError(f"Missing files in {model_dir}: {', '.join(missing)}")

config = json.loads((model_dir / "threshold_config.json").read_text(encoding="utf-8"))
threshold = float(config["optimal_threshold"])
max_length = int(config["max_length"])
if not 0 <= threshold <= 1 or max_length < 1:
    raise ValueError("The model export has an invalid decision threshold or token limit.")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer = AutoTokenizer.from_pretrained(model_dir, local_files_only=True)
model = AutoModelForSequenceClassification.from_pretrained(model_dir, local_files_only=True)
if model.config.num_labels != 2:
    raise ValueError("This notebook expects the fine-tuned two-label HalluGuard checkpoint.")
model.to(device).eval()


@torch.inference_mode()
def check_claim(premise, claim):
    if not premise.strip() or not claim.strip():
        raise ValueError("Enter both a passage and a claim.")
    inputs = tokenizer(
        premise.strip(),
        claim.strip(),
        return_tensors="pt",
        truncation=True,
        max_length=max_length,
    ).to(device)
    score = torch.softmax(model(**inputs).logits.float(), dim=-1)[0, 1].item()
    verdict = "Supported" if score >= threshold else "Unsupported"
    return verdict, score


print(f"HalluGuard is ready on {device}.")
print(f"Model folder: {model_dir.resolve()}")
print(f"Decision threshold: {threshold:.2f}")

### Try the sample pairs

The expected judgments below are based on reading the passages. Run the cell to compare them with the model's predictions.

In [ ]:
samples = [
    ("Date match", "Napoleon was defeated at Waterloo in 1815.",
     "Napoleon was defeated at Waterloo in 1815.", "Supported"),
    ("Date contradiction", "Napoleon was defeated at Waterloo in 1815.",
     "Napoleon was defeated at Waterloo in 1820.", "Unsupported"),
    ("Paraphrase", "The library will be closed to visitors on Monday.",
     "Visitors cannot enter the library on Monday.", "Supported"),
    ("Quantity contradiction", "The science club ordered 12 microscopes.",
     "The science club ordered 20 microscopes.", "Unsupported"),
    ("Missing evidence", "The science club ordered 12 microscopes.",
     "Each microscope cost 5,000 rupees.", "Unsupported"),
    ("Partial support", "The museum opened in May 2018.",
     "The museum opened in May 2018 and admission was free.", "Unsupported"),
]

rows = []
for example, premise, claim, expected in samples:
    verdict, score = check_claim(premise, claim)
    rows.append({
        "Example": example,
        "Passage": premise,
        "Claim": claim,
        "Expected": expected,
        "Prediction": verdict,
        "Support score": f"{score:.3f}",
    })

display(pd.DataFrame(rows))

### Check your own claim

Change the two strings and rerun this cell. The verdict is about the passage you supplied.

In [ ]:
premise = "Napoleon was defeated at Waterloo in 1815."
claim = "Napoleon was defeated at Waterloo in 1820."

verdict, score = check_claim(premise, claim)
print(f"{verdict} by the passage")
print(f"Support score: {score:.3f}  |  Decision threshold: {threshold:.2f}")

### Optional model-only export

For maintainers. If the attached model version also contains training results, this cell copies the five inference files into a new folder without retraining. It also carries over license and notice files found beside the weights. Preserve any upstream notices before sharing the copied weights. The model is about 1.75 GB, so enable this only when you need a new copy. On Kaggle, save and run the notebook with the switch enabled to capture the folder in a saved version's **Output**.

In [ ]:
from shutil import copy2
from tempfile import TemporaryDirectory

EXPORT_MODEL = False

if EXPORT_MODEL:
    output_root = Path("/kaggle/working") if Path("/kaggle/working").is_dir() else Path.cwd()
    export_dir = output_root / "halluguard_inference_export"
    if export_dir.exists():
        raise FileExistsError(f"An export already exists at {export_dir}.")

    notices = ("LICENSE", "LICENSE.txt", "NOTICE", "NOTICE.txt")
    with TemporaryDirectory(prefix=".halluguard-", dir=output_root) as staging:
        staged_model = Path(staging) / export_dir.name
        staged_model.mkdir()
        for name in INFERENCE_FILES:
            copy2(model_dir / name, staged_model / name)
        for name in notices:
            if (model_dir / name).is_file():
                copy2(model_dir / name, staged_model / name)
        staged_model.rename(export_dir)

    print(f"Inference-only export: {export_dir}")
else:
    print("Model export is disabled.")